In [1]:
import allel
import zarr
import xarray as xr
import plotly.express as px
import dask.array as da
from collections import Counter
import numpy as np
import json
import hashlib
import numba
import pandas as pd
import bokeh.plotting as bkplt
import bokeh.io as bkio
import bokeh.palettes as bokpalet
import bokeh.models as bkmod
import matplotlib.pyplot as plt
import hmmlearn
from Bio import SeqIO
from pathlib import Path
import malariagen_data

In [2]:
amin1 = malariagen_data.Amin1()#results_cache='~/amin-cache/')
df_samples = amin1.sample_metadata()

In [3]:
df_samples

,sample_id,derived_sample_id,partner_sample_id,contributor,country,location,year,month,latitude,longitude,...,admin1_name,admin1_iso,admin2_name,taxon_y,cohort_admin1_year,cohort_admin1_month,cohort_admin1_quarter,cohort_admin2_year,cohort_admin2_month,cohort_admin2_quarter
0,VBS09378-4248STDY7308980,VBS09378-4248STDY7308980,CB-2-00264,Brandy St. Laurent,Cambodia,Preah Kleang,2016,3,13.667,104.982,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,VBS09382-4248STDY7308981,VBS09382-4248STDY7308981,CB-2-00258,Brandy St. Laurent,Cambodia,Preah Kleang,2016,3,13.667,104.982,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,VBS09397-4248STDY7308982,VBS09397-4248STDY7308982,CB-2-00384,Brandy St. Laurent,Cambodia,Preah Kleang,2016,3,13.667,104.982,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,VBS09460-4248STDY7308986,VBS09460-4248STDY7308986,CB-2-02960,Brandy St. Laurent,Cambodia,Preah Kleang,2016,6,13.667,104.982,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,VBS09466-4248STDY7308989,VBS09466-4248STDY7308989,CB-2-04070,Brandy St. Laurent,Cambodia,Preah Kleang,2016,11,13.667,104.982,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
297,VBS16624-4248STDY7918667,VBS16624-4248STDY7918667,KV-32-01591,Brandy St. Laurent,Cambodia,Sayas,2014,6,13.548,107.025,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
298,VBS16625-4248STDY7918668,VBS16625-4248STDY7918668,KV-32-01499,Brandy St. Laurent,Cambodia,Sayas,2014,6,13.548,107.025,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
299,VBS16626-4248STDY7918669,VBS16626-4248STDY7918669,KV-32-01465,Brandy St. Laurent,Cambodia,Sayas,2014,6,13.548,107.025,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
300,VBS16628-4248STDY7918670,VBS16628-4248STDY7918670,KV-32-01454,Brandy St. Laurent,Cambodia,Sayas,2014,6,13.548,107.025,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
# Hashing func
def hash_params(*args, **kwargs):
    """Helper function to hash analysis parameters."""
    o = {
        'args': args,
        'kwargs': kwargs
    }
    s = json.dumps(o, sort_keys=True).encode()
    h = hashlib.md5(s).hexdigest()
    return h

def infer_roh(ind, chrom, analysis_name, results_dir):

    # Construct a key to save the results under
    results_key = hash_params(
        ind=ind,
        chrom=chrom,
        analysis_name=analysis_name
    )

    # Define paths for results files
    data_path = f'{results_dir}/{results_key}-roh.csv'

    try:
        # Try to load previously generated results
        df_roh = pd.read_csv(data_path)
        return df_roh
    except FileNotFoundError:
        # No previous results available, need to run analysis
        print(f'running analysis: {results_key}')


    ds = amin1.snp_calls(region = chrom, sample_query = f'partner_sample_id == "{ind}"')

    gt = ds['call_genotype'].compute()[:,0]
    # Load data

    pos = ds['variant_position'].compute()

    # Infer ROH for ind / chrom
    print(f'computing ROH for {ind}, {chrom}')
    df_roh = allel.roh_mhmm(gv=gt, pos=pos, contig_size = int(np.max(pos)))[0]
    df_roh['ind'] = ind
    df_roh['chrom'] = chrom
    
    
    # Save results to hash cache
    df_roh.to_csv(data_path, index=False)
    print(f'saved results: {results_key}')
    return(df_roh)

In [ ]:
rohlist = []
for sample in df_samples['partner_sample_id'].to_list():
    for contig in amin1.contigs:
        roh = infer_roh(ind=sample, chrom=contig,analysis_name = 'amin1_roh',results_dir='~/amin_cache/roh_v1_analysis/')
        rohlist.append(roh)
    

running analysis: f2851271a21dae321d6776ee2ead3d13
computing ROH for KV-31-03653, KB663832
saved results: f2851271a21dae321d6776ee2ead3d13
running analysis: 3aef0fa1643cdc6dd56338b40458d98b
computing ROH for KV-31-03653, KB663833
saved results: 3aef0fa1643cdc6dd56338b40458d98b
running analysis: 981dc8936ceae41ba651eea32ed15803
computing ROH for KV-31-03653, KB663844
saved results: 981dc8936ceae41ba651eea32ed15803
running analysis: 514aa7dd3e5f1991a502ba8e98149e0e
computing ROH for KV-31-03653, KB663855
saved results: 514aa7dd3e5f1991a502ba8e98149e0e
running analysis: 1f0e47232841d15438cbc96a5f46e42b
computing ROH for KV-31-03653, KB663866
saved results: 1f0e47232841d15438cbc96a5f46e42b
running analysis: 4c64d8e90f4c43134e5244da55663275
computing ROH for KV-31-03653, KB663877
saved results: 4c64d8e90f4c43134e5244da55663275
running analysis: 1d91fb6124d66b58971ade2c7ec53be7
computing ROH for KV-31-03653, KB663888
saved results: 1d91fb6124d66b58971ade2c7ec53be7
running analysis: d54ca48a4

In [ ]:
ds = amin1.snp_calls(region = 'KB663610', sample_query = f'partner_sample_id == "CB-2-00264"')


In [ ]:
gt = ds['call_genotype'].compute()[:,0]
# Load data

pos = ds['variant_position'].compute()

In [ ]:
df_roh = allel.roh_mhmm(gv=gt, pos=pos, contig_size = int(np.max(pos)))


In [ ]:
df_roh[0]

,start,stop,length,is_marginal
0,1,51313,51313,True
1,57605,74811,17207,False
2,610810,616846,6037,False
3,864676,871384,6709,False
4,958533,969497,10965,False
...,...,...,...,...
211,30817899,30863320,45422,False
212,30865346,30943830,78485,False
213,31035226,31041903,6678,False
214,31183042,31188309,5268,False
